# Leather Defect Multiclass Segmentation — Low-Light Robust UNet

**Dataset**: Processed leather defect dataset with normal + low-light (gamma-corrected) variants.

**Key features:**
1. **Multiclass segmentation** — 6 classes: background, color, cut, fold, glue, poke
2. **Low-light robustness** — trains on both normal and gamma-degraded images
3. **CLAHE preprocessing** — enhances contrast in low-light regions
4. **RGB input** (3 channels) — critical for detecting color defects
5. **Class-aware oversampling** — rare defects repeated more during training
6. **Per-class weighted Focal + Dice loss** — handles extreme class imbalance
7. **Attention UNet with Deep Supervision** — multi-scale learning
8. **50 epochs** with cosine annealing LR on Colab GPU

In [ ]:
# Mount Google Drive (Colab)
import os
IS_COLAB = os.path.exists('/content')
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
else:
    print('Running locally.')

## 1. Imports & Configuration

In [ ]:
import os, sys, warnings, glob, json
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras import backend as K

warnings.filterwarnings('ignore')

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

# ============================================================================
# CONFIGURATION
# ============================================================================
if IS_COLAB:
    DATA_DIR    = Path('/content/drive/MyDrive/image/leather_project/processed_dataset')
    RESULTS_DIR = Path('/content/drive/MyDrive/image/leather_project/unet_results_v3')
else:
    DATA_DIR    = Path(r'C:\Users\User\Downloads\lowlight\processed_dataset')
    RESULTS_DIR = Path(r'd:\7th sem\Image processing and CV\Leather_defect_detection\unet_results_v3')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE     = (256, 256)  # Native resolution of .npy files
IMG_CHANNELS = 3           # RGB
DEFECT_TYPES = ['color', 'cut', 'fold', 'glue', 'poke']
CLASS_NAMES  = ['background'] + DEFECT_TYPES  # 6 classes
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_MAP    = {dt: i+1 for i, dt in enumerate(DEFECT_TYPES)}  # color→1, cut→2, ...

BASE_FILTERS  = 32
BATCH_SIZE    = 8
EPOCHS        = 50
LEARNING_RATE = 3e-4
WARMUP_EPOCHS = 5
SEED          = 42
FOCAL_GAMMA   = 3.0
DICE_SMOOTH   = 1.0

CLASS_COLORS = np.array([
    [0,   0,   0  ],  # background
    [255, 0,   0  ],  # color — red
    [0,   255, 0  ],  # cut   — green
    [0,   0,   255],  # fold  — blue
    [255, 255, 0  ],  # glue  — yellow
    [255, 0,   255],  # poke  — magenta
], dtype=np.uint8)

# Oversampling multipliers for rare classes
OVERSAMPLE = {'color': 3, 'cut': 3, 'fold': 3, 'glue': 3, 'poke': 4, 'good': 1}

print(f'Data dir:    {DATA_DIR}')
print(f'Results dir: {RESULTS_DIR}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Class map:   {CLASS_MAP}')

## 2. Data Discovery

Build sample list from the `.npy` dataset. Each sample is `(image_path, mask_path, class_id)`.  
Low-light variants share the same mask as their parent image.

In [ ]:
def parse_defect_type(filename):
    """Extract defect type from filename like 'color_001.npy' or 'color_001_ll0.npy'."""
    stem = filename.replace('.npy', '')
    # Remove lowlight suffix if present (e.g., '_ll0', '_ll1')
    for suffix in ['_ll0', '_ll1', '_ll2', '_ll3']:
        stem = stem.replace(suffix, '')
    # The defect type is everything before the last underscore + number
    parts = stem.rsplit('_', 1)
    return parts[0] if len(parts) == 2 else stem


def discover_samples(split):
    """Discover all samples for a split (train/val/test).
    Returns list of (image_path, mask_path, class_id) tuples."""
    samples = []
    img_dir  = DATA_DIR / split / 'images'
    ll_dir   = DATA_DIR / split / 'lowlight'
    mask_dir = DATA_DIR / split / 'masks'

    # Normal images
    for img_path in sorted(img_dir.glob('*.npy')):
        mask_path = mask_dir / img_path.name
        if not mask_path.exists():
            continue
        dtype = parse_defect_type(img_path.name)
        class_id = CLASS_MAP.get(dtype, 0)  # 0 for 'good'
        samples.append((str(img_path), str(mask_path), class_id))

    # Low-light images (share parent mask)
    if ll_dir.exists():
        for ll_path in sorted(ll_dir.glob('*.npy')):
            # Derive parent name: color_001_ll0.npy → color_001.npy
            stem = ll_path.stem
            for suffix in ['_ll0', '_ll1', '_ll2', '_ll3']:
                stem = stem.replace(suffix, '')
            parent_mask = mask_dir / f'{stem}.npy'
            if not parent_mask.exists():
                continue
            dtype = parse_defect_type(ll_path.name)
            class_id = CLASS_MAP.get(dtype, 0)
            samples.append((str(ll_path), str(parent_mask), class_id))

    return samples


train_samples = discover_samples('train')
val_samples   = discover_samples('val')
test_samples  = discover_samples('test')

for name, slist in [('Train', train_samples), ('Val', val_samples), ('Test', test_samples)]:
    counts = Counter(CLASS_NAMES[s[2]] for s in slist)
    print(f'{name}: {len(slist)} samples — {dict(counts)}')

## 3. Data Loading with CLAHE + Augmentation

- **CLAHE** on the L channel of LAB space to recover contrast from low-light degradation
- **Strong augmentation** including HSV jitter for colour-defect robustness
- `.npy` files are float32 RGB [0,1], masks are float32 binary {0,1}

In [ ]:
def apply_clahe_rgb(img_float, clip_limit=3.0, tile_size=(8, 8)):
    """Apply CLAHE to a float32 RGB image [0,1]."""
    img_uint8 = (np.clip(img_float, 0, 1) * 255).astype(np.uint8)
    lab = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return enhanced.astype(np.float32) / 255.0


def load_sample(img_path, mask_path, class_id):
    """Load .npy image + mask, apply CLAHE, create multiclass mask."""
    img  = np.load(img_path).astype(np.float32)   # (256, 256, 3) RGB [0,1]
    mask = np.load(mask_path).astype(np.float32)   # (256, 256) binary {0,1}

    # Apply CLAHE to enhance low-light contrast
    img = apply_clahe_rgb(img)

    # Convert binary mask to multiclass
    mc_mask = np.zeros(IMG_SIZE, dtype=np.int32)
    mc_mask[mask > 0.5] = class_id  # 0 for good images (class_id=0)

    return img, mc_mask


def augment_pair(image, mask):
    """Strong augmentation for RGB images + mask."""
    # --- Geometric ---
    if np.random.rand() > 0.5:
        image, mask = np.fliplr(image).copy(), np.fliplr(mask).copy()
    if np.random.rand() > 0.5:
        image, mask = np.flipud(image).copy(), np.flipud(mask).copy()
    k = np.random.randint(0, 4)
    image, mask = np.rot90(image, k).copy(), np.rot90(mask, k).copy()

    # Random scale (0.8–1.2)
    if np.random.rand() > 0.4:
        scale = np.random.uniform(0.8, 1.2)
        h, w = image.shape[:2]
        nh, nw = int(h * scale), int(w * scale)
        img_s  = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_LINEAR)
        mask_s = cv2.resize(mask.astype(np.float32), (nw, nh),
                            interpolation=cv2.INTER_NEAREST).astype(np.int32)
        if scale > 1.0:
            sh, sw = (nh - h) // 2, (nw - w) // 2
            image, mask = img_s[sh:sh+h, sw:sw+w], mask_s[sh:sh+h, sw:sw+w]
        else:
            ph, pw = (h - nh) // 2, (w - nw) // 2
            img_new  = np.zeros_like(image)
            mask_new = np.zeros((h, w), dtype=np.int32)
            img_new[ph:ph+nh, pw:pw+nw]  = img_s
            mask_new[ph:ph+nh, pw:pw+nw] = mask_s
            image, mask = img_new, mask_new

    # --- Photometric ---
    # Brightness
    if np.random.rand() > 0.5:
        image = np.clip(image * np.random.uniform(0.6, 1.4), 0.0, 1.0)

    # Contrast
    if np.random.rand() > 0.5:
        f = np.random.uniform(0.6, 1.4)
        mean = np.mean(image, axis=(0, 1), keepdims=True)
        image = np.clip((image - mean) * f + mean, 0.0, 1.0)

    # HSV jitter — key for colour-defect robustness
    if np.random.rand() > 0.5:
        img_u8 = (image * 255).astype(np.uint8)
        hsv = cv2.cvtColor(img_u8, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] = (hsv[:, :, 0] + np.random.uniform(-10, 10)) % 180
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * np.random.uniform(0.7, 1.3), 0, 255)
        img_u8 = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        image = img_u8.astype(np.float32) / 255.0

    # Simulate additional low-light with random gamma
    if np.random.rand() > 0.6:
        gamma = np.random.uniform(1.0, 2.0)
        image = np.clip(np.power(image, gamma), 0.0, 1.0)

    # Gaussian noise
    if np.random.rand() > 0.5:
        noise = np.random.normal(0, 0.015, image.shape).astype(np.float32)
        image = np.clip(image + noise, 0.0, 1.0)

    # Gaussian blur
    if np.random.rand() > 0.7:
        ksize = np.random.choice([3, 5])
        image = cv2.GaussianBlur(image, (ksize, ksize), 0)

    return image.astype(np.float32), mask.astype(np.int32)


def create_dataset(samples, augment=False, repeat=True):
    """Create tf.data.Dataset from sample list."""
    img_paths  = [s[0] for s in samples]
    mask_paths = [s[1] for s in samples]
    class_ids  = [s[2] for s in samples]

    def gen():
        indices = list(range(len(img_paths)))
        if augment:
            np.random.shuffle(indices)
        for idx in indices:
            img, mask = load_sample(img_paths[idx], mask_paths[idx], class_ids[idx])
            if augment:
                img, mask = augment_pair(img, mask)
            yield img, mask

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec((IMG_SIZE[0], IMG_SIZE[1], IMG_CHANNELS), tf.float32),
            tf.TensorSpec((IMG_SIZE[0], IMG_SIZE[1]),               tf.int32),
        )
    )
    if repeat:
        ds = ds.repeat()
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


print('Data pipeline defined.')

### Visualise CLAHE Effect on Low-Light Images

In [ ]:
# Pick a low-light sample to visualise CLAHE
ll_samples = [s for s in train_samples if '_ll' in s[0]]
normal_samples = [s for s in train_samples if '_ll' not in s[0] and s[2] > 0]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for row, (sample, label) in enumerate([(normal_samples[0], 'Normal'), (ll_samples[0], 'Low-Light')]):
    raw = np.load(sample[0])
    clahe_img = apply_clahe_rgb(raw)
    mask = np.load(sample[1])
    mc_mask = np.zeros(IMG_SIZE, dtype=np.int32)
    mc_mask[mask > 0.5] = sample[2]

    axes[row, 0].imshow(raw)
    axes[row, 0].set_title(f'{label} — Original\nmean={raw.mean():.3f}', fontsize=11)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(clahe_img)
    axes[row, 1].set_title(f'{label} — After CLAHE\nmean={clahe_img.mean():.3f}', fontsize=11)
    axes[row, 1].axis('off')

    # Difference map
    diff = np.abs(clahe_img - raw)
    axes[row, 2].imshow(diff / diff.max() if diff.max() > 0 else diff)
    axes[row, 2].set_title('|CLAHE − Original|', fontsize=11)
    axes[row, 2].axis('off')

    cmap = ListedColormap(CLASS_COLORS / 255.0)
    axes[row, 3].imshow(mc_mask, cmap=cmap, vmin=0, vmax=NUM_CLASSES-1)
    axes[row, 3].set_title(f'Mask ({CLASS_NAMES[sample[2]]})', fontsize=11)
    axes[row, 3].axis('off')

plt.suptitle('CLAHE Preprocessing: Normal vs Low-Light', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'clahe_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Class Weights & Oversampled Datasets

In [ ]:
def compute_class_weights(samples):
    """Compute inverse-frequency class weights."""
    print('Computing class weights...')
    pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    for img_path, mask_path, class_id in samples:
        mask = np.load(mask_path)
        bg_pixels = np.sum(mask <= 0.5)
        fg_pixels = np.sum(mask > 0.5)
        pixel_counts[0] += bg_pixels
        if class_id > 0:
            pixel_counts[class_id] += fg_pixels

    total = pixel_counts.sum()
    for i, name in enumerate(CLASS_NAMES):
        pct = pixel_counts[i] / total * 100 if total > 0 else 0
        print(f'  {name}: {pixel_counts[i]:>10,} ({pct:.3f}%)')

    weights = np.ones(NUM_CLASSES, dtype=np.float32)
    for i in range(NUM_CLASSES):
        if pixel_counts[i] > 0:
            weights[i] = total / (NUM_CLASSES * pixel_counts[i])
    weights = np.clip(weights, 0.01, 200.0)
    weights /= weights.mean()

    print('  Normalised weights:')
    for i, name in enumerate(CLASS_NAMES):
        print(f'    {name}: {weights[i]:.4f}')
    return weights


# Use only unique normal images for weight computation (no lowlight duplicates)
normal_train = [s for s in train_samples if '_ll' not in s[0]]
class_weights = compute_class_weights(normal_train)

# Oversample training data
def oversample(samples):
    out = []
    for s in samples:
        dtype = parse_defect_type(os.path.basename(s[0]))
        repeat = OVERSAMPLE.get(dtype, 1)
        out.extend([s] * repeat)
    np.random.seed(SEED)
    np.random.shuffle(out)
    return out

train_oversampled = oversample(train_samples)
print(f'\nTrain: {len(train_samples)} → oversampled: {len(train_oversampled)}')
print(f'Val:   {len(val_samples)}')
print(f'Test:  {len(test_samples)}')

# Build datasets
train_ds = create_dataset(train_oversampled, augment=True,  repeat=True)
val_ds   = create_dataset(val_samples,       augment=False, repeat=True)

steps_per_epoch = max(len(train_oversampled) * 2 // BATCH_SIZE, 1)
val_steps       = max(len(val_samples)           // BATCH_SIZE, 1)
print(f'Steps/epoch: {steps_per_epoch} | Val steps: {val_steps}')

## 5. Attention UNet with Deep Supervision

In [ ]:
def squeeze_excite(x, ratio=8):
    f = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(f // ratio, 1), activation='relu')(se)
    se = layers.Dense(f, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, f))(se)
    return layers.Multiply()([x, se])


def conv_block(x, filters, drop=0.1):
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = squeeze_excite(x)
    if drop > 0:
        x = layers.SpatialDropout2D(drop)(x)
    return x


def attn_gate(skip, gate, f):
    tx = layers.Conv2D(f, 1, padding='same')(skip)
    tg = layers.Conv2D(f, 1, padding='same')(gate)
    a  = layers.Activation('relu')(layers.Add()([tx, tg]))
    a  = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(a)
    return layers.Multiply()([skip, a])


def build_attention_unet(input_shape, num_classes, bf=32):
    inp = layers.Input(shape=input_shape, name='input_image')

    # Encoder
    e1 = conv_block(inp, bf, drop=0.05)
    e2 = conv_block(layers.MaxPooling2D(2)(e1), bf*2, drop=0.10)
    e3 = conv_block(layers.MaxPooling2D(2)(e2), bf*4, drop=0.15)
    e4 = conv_block(layers.MaxPooling2D(2)(e3), bf*8, drop=0.20)

    # Bottleneck
    b = conv_block(layers.MaxPooling2D(2)(e4), bf*16, drop=0.30)

    # Decoder
    u4 = layers.Conv2DTranspose(bf*8, 2, strides=2, padding='same')(b)
    d4 = conv_block(layers.Concatenate()([u4, attn_gate(e4, u4, bf*4)]), bf*8, drop=0.20)

    u3 = layers.Conv2DTranspose(bf*4, 2, strides=2, padding='same')(d4)
    d3 = conv_block(layers.Concatenate()([u3, attn_gate(e3, u3, bf*2)]), bf*4, drop=0.15)

    # Auxiliary output (deep supervision) from level 3
    aux = layers.Conv2D(num_classes, 1, activation='softmax', name='aux_output')(
        layers.UpSampling2D(size=(4, 4), interpolation='bilinear')(d3)
    )

    u2 = layers.Conv2DTranspose(bf*2, 2, strides=2, padding='same')(d3)
    d2 = conv_block(layers.Concatenate()([u2, attn_gate(e2, u2, bf)]), bf*2, drop=0.10)

    u1 = layers.Conv2DTranspose(bf, 2, strides=2, padding='same')(d2)
    d1 = conv_block(layers.Concatenate()([u1, attn_gate(e1, u1, bf//2)]), bf, drop=0.05)

    out = layers.Conv2D(num_classes, 1, activation='softmax', name='main_output')(d1)

    return Model(inp, [out, aux], name='attention_unet_deep_sup')


print('Architecture defined.')

## 6. Per-Class Weighted Loss & Metrics

In [ ]:
tf_cw = tf.constant(class_weights, dtype=tf.float32)


def weighted_focal_loss(y_true, y_pred):
    y_t = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_p = tf.clip_by_value(tf.reshape(y_pred, [-1, NUM_CLASSES]), 1e-7, 1.0 - 1e-7)
    oh  = tf.one_hot(y_t, NUM_CLASSES)
    ce  = -oh * tf.math.log(y_p)
    pt  = tf.reduce_sum(oh * y_p, axis=-1)
    pw  = tf.gather(tf_cw, y_t)
    fw  = pw * tf.pow(1.0 - pt, FOCAL_GAMMA)
    return tf.reduce_mean(tf.reduce_sum(ce, axis=-1) * fw)


def weighted_dice_loss(y_true, y_pred):
    y_t = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_p = tf.reshape(y_pred, [-1, NUM_CLASSES])
    oh  = tf.one_hot(y_t, NUM_CLASSES)
    inter = tf.reduce_sum(oh * y_p, axis=0)
    union = tf.reduce_sum(oh, axis=0) + tf.reduce_sum(y_p, axis=0)
    dice  = (2.0 * inter + DICE_SMOOTH) / (union + DICE_SMOOTH)
    # Weight defect classes, skip background
    dw = tf_cw[1:] / tf.reduce_sum(tf_cw[1:])
    return 1.0 - tf.reduce_sum(dice[1:] * dw)


def combined_loss(y_true, y_pred):
    return 0.3 * weighted_focal_loss(y_true, y_pred) + 0.7 * weighted_dice_loss(y_true, y_pred)


def mean_iou_metric(y_true, y_pred):
    y_t = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_p = tf.cast(tf.argmax(tf.reshape(y_pred, [-1, NUM_CLASSES]), axis=-1), tf.int32)
    cm  = tf.cast(tf.math.confusion_matrix(y_t, y_p, num_classes=NUM_CLASSES), tf.float32)
    d   = tf.linalg.diag_part(cm)
    den = tf.reduce_sum(cm, 1) + tf.reduce_sum(cm, 0) - d
    iou = tf.where(den > 0, d / den, tf.zeros_like(d))
    v   = tf.cast(den > 0, tf.float32)
    return tf.math.divide_no_nan(tf.reduce_sum(iou), tf.reduce_sum(v))


def mean_dice_metric(y_true, y_pred):
    y_t = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_p = tf.cast(tf.argmax(tf.reshape(y_pred, [-1, NUM_CLASSES]), axis=-1), tf.int32)
    cm  = tf.cast(tf.math.confusion_matrix(y_t, y_p, num_classes=NUM_CLASSES), tf.float32)
    d   = tf.linalg.diag_part(cm)
    den = tf.reduce_sum(cm, 1) + tf.reduce_sum(cm, 0)
    dice = tf.where(den > 0, 2.0 * d / den, tf.zeros_like(d))
    v    = tf.cast(den > 0, tf.float32)
    return tf.math.divide_no_nan(tf.reduce_sum(dice), tf.reduce_sum(v))


print('Loss & metrics defined.')

## 7. Callbacks & Helpers

In [ ]:
class CosineAnnealingWarmup(callbacks.Callback):
    def __init__(self, max_lr, warmup, total, min_lr=1e-6):
        super().__init__()
        self.max_lr, self.warmup, self.total, self.min_lr = max_lr, warmup, total, min_lr
        self.lr_history = []

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup:
            lr = self.max_lr * (epoch + 1) / max(self.warmup, 1)
        else:
            p = (epoch - self.warmup) / max(self.total - self.warmup, 1)
            lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + np.cos(np.pi * p))
        try:
            self.model.optimizer.learning_rate.assign(lr)
        except Exception:
            K.set_value(self.model.optimizer.learning_rate, lr)
        self.lr_history.append(lr)


def colorize_mask(mask):
    h, w = mask.shape
    c = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(NUM_CLASSES):
        c[mask == i] = CLASS_COLORS[i]
    return c


def visualize_predictions(model, samples_list, n=6, save_path=None, title='Predictions'):
    n = min(n, len(samples_list))
    fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    for i in range(n):
        img, gt = load_sample(*samples_list[i])
        preds = model.predict(np.expand_dims(img, 0), verbose=0)
        pred_main = preds[0] if isinstance(preds, list) else preds
        pm = np.argmax(pred_main[0], axis=-1)

        is_ll = '_ll' in samples_list[i][0]
        cls   = CLASS_NAMES[samples_list[i][2]]
        tag   = f'{cls} [{"LL" if is_ll else "Normal"}]'

        axes[i, 0].imshow(img); axes[i, 0].set_title(f'Input ({tag})', fontsize=10); axes[i, 0].axis('off')
        axes[i, 1].imshow(colorize_mask(gt)); axes[i, 1].set_title('Ground Truth', fontsize=10); axes[i, 1].axis('off')
        axes[i, 2].imshow(colorize_mask(pm)); axes[i, 2].set_title('Prediction', fontsize=10); axes[i, 2].axis('off')

        overlay = (img * 255).astype(np.uint8).copy()
        pc = colorize_mask(pm)
        w = pm > 0
        if w.any():
            overlay[w] = (0.5 * overlay[w] + 0.5 * pc[w]).astype(np.uint8)
        axes[i, 3].imshow(overlay); axes[i, 3].set_title('Overlay', fontsize=10); axes[i, 3].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_history(history, save_path=None):
    h = history.history
    # Auto-detect metric keys (handle deep supervision prefixes)
    def find_key(candidates):
        for k in candidates:
            if k in h:
                return k
        return None

    loss_k     = find_key(['main_output_loss', 'loss'])
    val_loss_k = find_key(['val_main_output_loss', 'val_loss'])
    iou_k      = find_key(['main_output_mean_iou_metric', 'mean_iou_metric'])
    val_iou_k  = find_key(['val_main_output_mean_iou_metric', 'val_mean_iou_metric'])
    dice_k     = find_key(['main_output_mean_dice_metric', 'mean_dice_metric'])
    val_dice_k = find_key(['val_main_output_mean_dice_metric', 'val_mean_dice_metric'])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    if loss_k:     axes[0].plot(h[loss_k],     label='Train', lw=2)
    if val_loss_k: axes[0].plot(h[val_loss_k], label='Val',   lw=2)
    axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

    if iou_k:     axes[1].plot(h[iou_k],     label='Train', lw=2)
    if val_iou_k: axes[1].plot(h[val_iou_k], label='Val',   lw=2)
    axes[1].set_title('Mean IoU', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

    if dice_k:     axes[2].plot(h[dice_k],     label='Train', lw=2)
    if val_dice_k: axes[2].plot(h[val_dice_k], label='Val',   lw=2)
    axes[2].set_title('Mean Dice', fontweight='bold'); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle('Training History', fontweight='bold', fontsize=14)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def evaluate_model(model, samples_list, label=''):
    print(f'\n{"="*60}')
    print(f'EVALUATION {label}')
    print('=' * 60)
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for s in samples_list:
        img, gt = load_sample(*s)
        preds = model.predict(np.expand_dims(img, 0), verbose=0)
        pred_main = preds[0] if isinstance(preds, list) else preds
        pm = np.argmax(pred_main[0], axis=-1)
        for tc in range(NUM_CLASSES):
            for pc in range(NUM_CLASSES):
                cm[tc, pc] += np.sum((gt == tc) & (pm == pc))

    print(f"\n{'Class':<12} {'IoU':>8} {'Dice':>8} {'Prec':>8} {'Recall':>8}")
    print('-' * 48)
    ious, dices = [], []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        iou  = tp / (tp+fp+fn) if (tp+fp+fn) > 0 else 0.0
        dice = 2*tp / (2*tp+fp+fn) if (2*tp+fp+fn) > 0 else 0.0
        prec = tp / (tp+fp) if (tp+fp) > 0 else 0.0
        rec  = tp / (tp+fn) if (tp+fn) > 0 else 0.0
        ious.append(iou); dices.append(dice)
        print(f'{CLASS_NAMES[c]:<12} {iou:>8.4f} {dice:>8.4f} {prec:>8.4f} {rec:>8.4f}')
    print('-' * 48)
    print(f'{"Mean(all)":<12} {np.mean(ious):>8.4f} {np.mean(dices):>8.4f}')
    print(f'{"Mean(defect)":<12} {np.mean(ious[1:]):>8.4f} {np.mean(dices[1:]):>8.4f}')
    return ious, dices


print('Helpers defined.')

## 8. Build & Compile Model

In [ ]:
tf.keras.backend.clear_session()

model = build_attention_unet(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], IMG_CHANNELS),
    num_classes=NUM_CLASSES,
    bf=BASE_FILTERS,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss={'main_output': combined_loss, 'aux_output': combined_loss},
    loss_weights={'main_output': 1.0, 'aux_output': 0.4},
    metrics={'main_output': [mean_iou_metric, mean_dice_metric]},
)

print(f'Model: {model.name}')
print(f'Parameters: {model.count_params():,}')
model.summary()

## 9. Train

In [ ]:
model_path = str(RESULTS_DIR / 'best_unet_v3.keras')

# Deep supervision wrapper
def ds_wrap(ds):
    return ds.map(lambda x, y: (x, {'main_output': y, 'aux_output': y}),
                  num_parallel_calls=tf.data.AUTOTUNE)

train_ds_deep = ds_wrap(train_ds)
val_ds_deep   = ds_wrap(val_ds)

monitor = 'val_main_output_mean_iou_metric'

cb_list = [
    callbacks.ModelCheckpoint(model_path, monitor=monitor, mode='max',
                              save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor=monitor, mode='max',
                            patience=20, restore_best_weights=True, verbose=1),
    CosineAnnealingWarmup(LEARNING_RATE, WARMUP_EPOCHS, EPOCHS),
]

print('=' * 60)
print(f'Training: {EPOCHS} epochs | Batch: {BATCH_SIZE} | Steps: {steps_per_epoch}')
print(f'LR: {LEARNING_RATE} | Loss: 0.3×Focal + 0.7×Dice (class-weighted)')
print(f'Input: {IMG_SIZE} × {IMG_CHANNELS}ch (RGB + CLAHE) | Classes: {NUM_CLASSES}')
print(f'Data: {len(train_oversampled)} train (oversampled) + {len(val_samples)} val')
print('=' * 60)

history = model.fit(
    train_ds_deep,
    validation_data=val_ds_deep,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    callbacks=cb_list,
    verbose=1,
)

plot_history(history, save_path=str(RESULTS_DIR / 'training_history.png'))

## 10. Evaluate & Visualise

In [ ]:
# Load best model
print('Loading best model...')
if os.path.exists(model_path):
    model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            'combined_loss': combined_loss,
            'weighted_focal_loss': weighted_focal_loss,
            'weighted_dice_loss': weighted_dice_loss,
            'mean_iou_metric': mean_iou_metric,
            'mean_dice_metric': mean_dice_metric,
        }
    )
    print('Best model loaded.')

# Evaluate on ALL val samples
evaluate_model(model, val_samples, label='(All Val)')

# Evaluate separately: Normal vs Low-Light
val_normal = [s for s in val_samples if '_ll' not in s[0]]
val_ll     = [s for s in val_samples if '_ll' in s[0]]
if val_normal:
    evaluate_model(model, val_normal, label='(Val Normal Only)')
if val_ll:
    evaluate_model(model, val_ll, label='(Val Low-Light Only)')

# Evaluate on test set
evaluate_model(model, test_samples, label='(Test Set)')

In [ ]:
# Visualise: mix of normal + lowlight samples from val
vis_samples = []
for dtype in DEFECT_TYPES:
    normal = [s for s in val_samples if parse_defect_type(os.path.basename(s[0])) == dtype and '_ll' not in s[0]]
    ll     = [s for s in val_samples if parse_defect_type(os.path.basename(s[0])) == dtype and '_ll' in s[0]]
    if normal: vis_samples.append(normal[0])
    if ll:     vis_samples.append(ll[0])

visualize_predictions(
    model, vis_samples, n=len(vis_samples),
    save_path=str(RESULTS_DIR / 'val_predictions.png'),
    title='Validation: Normal vs Low-Light Predictions'
)

# Test set predictions
visualize_predictions(
    model, test_samples[:10], n=10,
    save_path=str(RESULTS_DIR / 'test_predictions.png'),
    title='Test Set Predictions'
)

print(f'\nAll results saved to: {RESULTS_DIR}')
print(f'Best model: {model_path}')

## 11. Download Model (Colab)

In [ ]:
if IS_COLAB:
    from google.colab import files
    if os.path.exists(model_path):
        files.download(model_path)
    else:
        print('Train the model first.')
else:
    print(f'Model at: {model_path}')